In [1]:
import poptorch
print(poptorch.__version__)
print("IPU hardware available:", poptorch.ipuHardwareIsAvailable())

3.3.0-960e9c294b
IPU hardware available: True


In [2]:
#Setting the experiment
MODEL_TYPE = "ResNet18"

# Datasets size & class balancing
WHOLE_DATA = False
BALANCED = False

# Data size handling for the IPU memory 
BATCH_SIZE = 1
REP_FACT   = 4
DEVICE_ITR = 5
GRAD_ACCUM = 2

# HPs for the model training
current_lr = 5e-5
current_decay = 1e-4

num_epochs = 10
patience = 5
lr_patience = 2

lr_factor = 0.1
min_lr = 1e-7


# Dataset Loading

In [24]:
# Loading the dataset
from datasets import load_dataset
from datasets import load_from_disk, concatenate_datasets
import os
import torch

my_dataset = "wltjr1007/Camelyon17-WILDS"
my_dataset_path = '/home/work/datasets/camelyon17_wilds'
my_dataset_path_train = '/home/work/datasets/camelyon17_wilds/train'

if os.path.exists(my_dataset_path) and os.listdir(my_dataset_path_train):
  print("Loading dataset from local disk...")
  dataset = load_from_disk(my_dataset_path)
else:
  print("Local data not found. Downloading from Hugging Face...")
  dataset = load_dataset(my_dataset, cache_dir=my_dataset_path)
  dataset.save_to_disk(my_dataset_path)
  print(f"Dataset saved to {my_dataset_path}")

Loading dataset from local disk...


In [4]:
sample = dataset["train"][0]
print(sample.keys())
print(type(sample["image"]))
print(sample["image"].size)
print(sample["image"].mode)

dict_keys(['image', 'label', 'center', 'image_id', 'patient', 'node', 'x_coord', 'y_coord', 'slide'])
<class 'PIL.PngImagePlugin.PngImageFile'>
(96, 96)
RGBA


In [5]:
# To convert HF dataset ultimately to Poptorch, one 🐉 at a time
import torch

class Camelyon17Dataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None, return_metadata=False):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]

        image = sample["image"].convert("RGB")  # RGBA -> RGB
        label = torch.tensor(sample["label"], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        if self.return_metadata:
            metadata = {
                "center": sample["center"],
                "image_id": sample["image_id"],
                "patient": sample["patient"],
                "node": sample["node"],
                "x_coord": sample["x_coord"],
                "y_coord": sample["y_coord"],
                "slide": sample["slide"],
            }
            return image, label, metadata

        return image, label

In [6]:
# PIL image to ImageNet normalized Tensor
from torchvision import transforms

my_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std =[0.229,0.224,0.225]
    )
])

# Dataset Preparation

In [7]:
# label structure
sample = dataset["train"].shuffle()

In [8]:
print("label =", sample["label"][:10])

labels = set()

for i in range(100):
    labels.add(sample[i]["label"])

print(labels)

label = [0, 1, 0, 0, 0, 1, 1, 1, 0, 1]
{0, 1}


In [9]:
from collections import Counter

counter = Counter(
    sample[i]["label"] for i in range(100)
)

print(counter)

Counter({1: 55, 0: 45})


In [10]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'center', 'image_id', 'patient', 'node', 'x_coord', 'y_coord', 'slide'],
        num_rows: 302436
    })
    validation: Dataset({
        features: ['image', 'label', 'center', 'image_id', 'patient', 'node', 'x_coord', 'y_coord', 'slide'],
        num_rows: 68464
    })
    test: Dataset({
        features: ['image', 'label', 'center', 'image_id', 'patient', 'node', 'x_coord', 'y_coord', 'slide'],
        num_rows: 85054
    })
})


In [11]:
def make_random_subset(hf_dataset, n_samples=None, seed=42):
    """
    Shuffle a Hugging Face dataset and optionally take the first n samples.
    Works for train/validation/test.
    """
    shuffled = hf_dataset.shuffle(seed=seed)

    if n_samples is None:
        return shuffled

    n_samples = min(n_samples, len(shuffled))
    return shuffled.select(range(n_samples))

def make_balanced_subset(hf_dataset, n_per_class=5000, seed=42, label_key="label"):
    shuffled = hf_dataset.shuffle(seed=seed)

    selected = {}
    indices = []

    for i, label in enumerate(shuffled[label_key]):
        label = int(label)

        if label not in selected:
            selected[label] = 0

        if selected[label] < n_per_class:
            indices.append(i)
            selected[label] += 1

        if all(v >= n_per_class for v in selected.values()) and len(selected) >= 2:
            break

    return shuffled.select(indices)

In [25]:
if (WHOLE_DATA == True):
    NO_TRAIN_DATA = len(dataset['train'])
    NO_TEST_DATA = len(dataset['test'])
    NO_VALID_DATA = len(dataset['validation'])  

else:

    if MODEL_TYPE == "ResNet18":
        NO_TRAIN_DATA = 20000
        NO_VALID_DATA = 10000
        NO_TEST_DATA = 10000
        
    elif MODEL_TYPE == "ResNet34":
        NO_TRAIN_DATA = 20000
        NO_VALID_DATA = 10000
        NO_TEST_DATA = 10000
        
    elif MODEL_TYPE == "ResNet50":
        NO_TRAIN_DATA = 20000
        NO_VALID_DATA = 10000
        NO_TEST_DATA = 10000
        
if (BALANCED == False):
    train_hf = make_random_subset(dataset["train"], n_samples=NO_TRAIN_DATA, seed=42)
    val_hf   = make_random_subset(dataset["validation"], n_samples=NO_VALID_DATA, seed=43)
    test_hf  = make_random_subset(dataset["test"], n_samples=NO_TEST_DATA, seed=44)
else:
    train_hf = make_balanced_subset(dataset["train"], n_per_class=NO_TRAIN_DATA, seed=42)
    val_hf   = make_balanced_subset(dataset["validation"], n_per_class=NO_VALID_DATA, seed=43)
    test_hf  = make_balanced_subset(dataset["test"], n_per_class=NO_TEST_DATA, seed=44)
    
print(NO_TRAIN_DATA, NO_VALID_DATA, NO_TEST_DATA)

20000 10000 10000


In [13]:
print(len(train_hf), len(val_hf), len(test_hf))

20000 10000 10000


In [14]:
from collections import Counter

for name, split in {
    "train": train_hf,
    "validation": val_hf,
    "test": test_hf
}.items():
    print(name, Counter(split["label"]))

train Counter({1: 10061, 0: 9939})
validation Counter({0: 5028, 1: 4972})
test Counter({1: 5059, 0: 4941})


In [15]:
#if (WHOLE_DATA == True):
#    save_dir = "/home/work/datasets/camelyon17_wilds/whole"
#else:
#    save_dir = "/home/work/datasets/camelyon17_wilds/subsets"

#train_hf.save_to_disk(f"{save_dir}/train")
#val_hf.save_to_disk(f"{save_dir}/val")
#test_hf.save_to_disk(f"{save_dir}/test")

In [16]:
#from datasets import load_from_disk

#train_hf = load_from_disk(f"{save_dir}/train")
#val_hf   = load_from_disk(f"{save_dir}/val")
#test_hf  = load_from_disk(f"{save_dir}/test")

# Model Definition

In [17]:
import torch
import torchvision.models as models

class ResNet18Camelyon(torch.nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        weights = (
            models.ResNet18_Weights.DEFAULT
            if pretrained else None
        )

        self.backbone = models.resnet18(weights=weights)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(in_features, num_classes)

    def forward(self, x, labels=None):
        logits = self.backbone(x)

        if labels is None:
            return logits

        loss = torch.nn.functional.cross_entropy(logits, labels)
        return logits, loss
    
class ResNet34Camelyon(torch.nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        weights = (
            models.ResNet34_Weights.DEFAULT
            if pretrained else None
        )

        self.backbone = models.resnet34(weights=weights)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(in_features, num_classes)

    def forward(self, x, labels=None):
        logits = self.backbone(x)

        if labels is None:
            return logits

        loss = torch.nn.functional.cross_entropy(logits, labels)
        return logits, loss
    
class ResNet50Camelyon(torch.nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        if pretrained:
            weights = models.ResNet50_Weights.IMAGENET1K_V2
        else:
            weights = None

        self.backbone = models.resnet50(weights=weights)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(in_features, num_classes)

    def forward(self, x, labels=None):
        logits = self.backbone(x)

        if labels is None:
            return logits

        loss = torch.nn.functional.cross_entropy(logits, labels)
        return logits, loss

## Dataloader Instantiation

In [26]:
import poptorch
import torch

batch_size = BATCH_SIZE

train_opts = poptorch.Options()
train_opts.replicationFactor(REP_FACT)
train_opts.deviceIterations(DEVICE_ITR)
train_opts.Training.gradientAccumulation(GRAD_ACCUM)

#print(train_opts)

# 2. Inference options
infer_opts = poptorch.Options()
infer_opts.replicationFactor(REP_FACT)
infer_opts.deviceIterations(GRAD_ACCUM)

#print(infer_opts)

train_data = Camelyon17Dataset(
    train_hf,
    transform=my_transform,
    return_metadata=False
)

train_loader = poptorch.DataLoader(
    train_opts,
    train_data,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=4
)

val_data = Camelyon17Dataset(
    val_hf,
    transform=my_transform,
    return_metadata=False
)

val_loader = poptorch.DataLoader(
    infer_opts,
    val_data,
    batch_size=10,
    shuffle=False,
    drop_last=True,
    num_workers=4
)

#val_loader_cpu = torch.utils.data.DataLoader(
#    val_data,
#    batch_size=10,
#    shuffle=False,
#    drop_last=False,
#    num_workers=4
#)

In [19]:
def cleanup_poptorch_wrappers():
    for name in ["poptorch_train", "poptorch_val", "infer_model", "display_infer_model"]:
        if name in globals():
            obj = globals()[name]

            try:
                print(name, "compiled:", obj.isCompiled())
            except Exception:
                pass

            try:
                obj.detachFromDevice()
                print(name, "detached")
            except Exception as e:
                print(name, "detach skipped:", e)

            try:
                obj.destroy()
                print(name, "destroyed")
            except Exception as e:
                print(name, "destroy skipped:", e)

#cleanup_poptorch_wrappers()

## Models instantiation

In [27]:
import copy

if MODEL_TYPE == "ResNet18":
    # 1. Two separate model objects
    train_model = ResNet18Camelyon(num_classes=2, pretrained=True)
    val_model   = ResNet18Camelyon(num_classes=2, pretrained=False)
    #display_model = ResNet18Camelyon(num_classes=2, pretrained=False)
    SAVE_MODEL_FILE = "./models/best_resnet18_camelyon17_wilds.pt"
elif MODEL_TYPE == "ResNet34":
    train_model = ResNet34Camelyon(num_classes=2, pretrained=True)
    val_model   = ResNet34Camelyon(num_classes=2, pretrained=False)
    #display_model = ResNet34Camelyon(num_classes=2, pretrained=False)
    SAVE_MODEL_FILE = "./models/best_resnet34_camelyon17_wilds.pt"
elif MODEL_TYPE == "ResNet50":
    train_model = ResNet50Camelyon(num_classes=2, pretrained=True)
    val_model   = ResNet50Camelyon(num_classes=2, pretrained=False)
    #display_model = ResNet50Camelyon(num_classes=2, pretrained=False)
    SAVE_MODEL_FILE = "./models/best_resnet50_camelyon17_wilds.pt"
    

# make sure val model starts identical
val_model.load_state_dict(train_model.state_dict())

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=current_lr,
    weight_decay=current_decay
)

# 2. Wrap separately
poptorch_train = poptorch.trainingModel(
    train_model,
    options=train_opts,
    optimizer=optimizer
)

poptorch_val = poptorch.inferenceModel(
    val_model,
    options=infer_opts
)

# Compile validation inference model once before the training loop
val_images0, val_labels0 = next(iter(val_loader))

with torch.no_grad():
    _ = poptorch_val(val_images0)

print("Validation IPU model compiled.")
print("Train compiled:", poptorch_train.isCompiled())
print("Validation compiled:", poptorch_val.isCompiled())

Graph compilation: 100%|██████████| 100/100 [01:28<00:00]


Validation IPU model compiled.
Train compiled: False
Validation compiled: True


## Validation function (atm called evaluation)

In [21]:
def evaluate(infer_model, data_loader):
    infer_model.eval()

    total = 0
    correct = 0
    total_loss = 0.0
    n_steps = 0

    with torch.no_grad():
        for images, labels in data_loader:
            logits = infer_model(images)

            loss = torch.nn.functional.cross_entropy(logits, labels)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            total_loss += loss.item()
            n_steps += 1

    acc = correct / total
    avg_loss = total_loss / n_steps

    return avg_loss, acc

def evaluate_cpu(model, data_loader):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0
    n_steps = 0

    with torch.no_grad():
        for images, labels in data_loader:
            logits = model(images)
            loss = torch.nn.functional.cross_entropy(logits, labels)

            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            total_loss += loss.item()
            n_steps += 1

    return total_loss / n_steps, correct / total

# Used for dual model training
def evaluate_ipu(infer_model, data_loader):
    total = 0
    correct = 0
    total_loss = 0.0
    n_steps = 0

    with torch.no_grad():
        for images, labels in data_loader:
            logits = infer_model(images)

            loss = torch.nn.functional.cross_entropy(logits, labels)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            total_loss += loss.item()
            n_steps += 1

    return total_loss / n_steps, correct / total

In [22]:
for obj_name in [
    "poptorch_train",
    "poptorch_val",
    "infer_model",
    "display_infer_model"
]:
    if obj_name in globals():
        obj = globals()[obj_name]
        try:
            obj.detachFromDevice()
            print(f"{obj_name}: detached")
        except Exception as e:
            print(f"{obj_name}: detach skipped:", e)

        try:
            obj.destroy()
            print(f"{obj_name}: destroyed")
        except Exception as e:
            print(f"{obj_name}: destroy skipped:", e)

poptorch_train: detach skipped: 'poptorch_py_error': Model has not been compiled or has been destroyed.
poptorch_train: destroyed
poptorch_val: detached
poptorch_val: destroyed


## Training Loop

In [28]:
import time

start_time = time.time()

best_val_loss = float("inf")
best_val_acc = 0.0

epochs_without_improvement = 0
epochs_without_lr_improvement = 0

for epoch in range(num_epochs):
    running_loss = 0.0
    n_steps = 0
    
    time0 = time.time()
    
    poptorch_train.train()

    for images, labels in train_loader:
        logits, loss = poptorch_train(images, labels)

        #running_loss += float(loss)
        if(n_steps == 0):
            print(loss, loss.shape)
        running_loss += loss.mean().item()
        n_steps += 1

    train_loss = running_loss / n_steps

    # 1. pull updated weights from training IPU to host train_model
    poptorch_train.copyWeightsToHost()

    # 2. copy host weights from train_model into val_model
    val_model.load_state_dict(train_model.state_dict())

    # 3. push updated val_model weights to validation IPU executable
    poptorch_val.copyWeightsToDevice()
    print("poptotch_val.copyWeightsToDevice() worked")

    # 4. validate on IPU without recompiling
    val_loss, val_acc = evaluate_ipu(poptorch_val, val_loader)

    epoch_time = time.time()-time0
    
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e} | "
        f"elapsed time={epoch_time:.1f} sec."
    )

    # checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        epochs_without_improvement = 0
        epochs_without_lr_improvement = 0
                                     
        # make sure host weights are current before saving
        poptorch_train.copyWeightsToHost()

        torch.save(train_model.state_dict(), SAVE_MODEL_FILE)
        print("Saved best model.")
    else:
        epochs_without_improvement += 1
        epochs_without_lr_improvement += 1

    # LR reduction on plateau
    if epochs_without_lr_improvement >= lr_patience:
        new_lr = max(current_lr * lr_factor, min_lr)

        if new_lr < current_lr:
            current_lr = new_lr
            print(f"Reducing LR to {current_lr:.2e}")

            # update existing optimizer LR
            for param_group in optimizer.param_groups:
                param_group["lr"] = current_lr

            # tell PopTorch to use the updated optimizer
            poptorch_train.setOptimizer(optimizer)

            epochs_without_lr_improvement = 0

    # early stopping
    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

print(f"Elasped time:{(time.time()-start_time)/60.0:.1f} min.")

Graph compilation: 100%|██████████| 100/100 [01:32<00:00]


tensor([1.1584, 0.3827, 0.3772, 0.3760]) torch.Size([4])
poptotch_val.copyWeightsToDevice() worked
Epoch 1/10 | train_loss=0.2676 | val_loss=0.4251 | val_acc=0.8430 | lr=5.00e-05 | elapsed time=123.6 sec.
Saved best model.
tensor([0.0260, 0.0262, 0.0495, 0.0084]) torch.Size([4])
poptotch_val.copyWeightsToDevice() worked
Epoch 2/10 | train_loss=0.0907 | val_loss=0.6584 | val_acc=0.8110 | lr=5.00e-05 | elapsed time=22.9 sec.
tensor([0.0027, 0.0112, 0.0030, 0.0038]) torch.Size([4])
poptotch_val.copyWeightsToDevice() worked
Epoch 3/10 | train_loss=0.0526 | val_loss=0.7285 | val_acc=0.7652 | lr=5.00e-05 | elapsed time=23.0 sec.
Reducing LR to 5.00e-06
tensor([0.0011, 0.0050, 0.0013, 0.0011]) torch.Size([4])
poptotch_val.copyWeightsToDevice() worked
Epoch 4/10 | train_loss=0.0129 | val_loss=0.6965 | val_acc=0.7510 | lr=5.00e-06 | elapsed time=23.0 sec.
tensor([0.0088, 0.0009, 0.0074, 0.0036]) torch.Size([4])
poptotch_val.copyWeightsToDevice() worked
Epoch 5/10 | train_loss=0.0084 | val_loss=

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(images.dtype)
print(labels.dtype)

## Evaluation with a fresh test model

In [29]:
# Finish training model
poptorch_train.copyWeightsToHost()

try:
    poptorch_train.detachFromDevice()
except Exception as e:
    print("detach warning:", e)

try:
    poptorch_train.destroy()
except Exception as e:
    print("destroy warning:", e)

In [30]:
#Do not wrap train_model again for inference.
#Use a fresh test_model loaded from the best checkpoint.
if MODEL_TYPE == "ResNet18":
    test_model = ResNet18Camelyon(num_classes=2, pretrained=False)
elif MODEL_TYPE == "ResNet34":
    test_model = ResNet34Camelyon(num_classes=2, pretrained=False)
elif MODEL_TYPE == "ResNet50":
    test_model = ResNet50Camelyon(num_classes=2, pretrained=False)

print(f"Loading the train model's weights from {SAVE_MODEL_FILE}")
test_model.load_state_dict(torch.load(SAVE_MODEL_FILE))
test_model.eval()

# opts should be set before instanciating the model and data loading so they can be shared in both tasks
REP_FACT   = 4
DEVICE_ITR = 1

infer_opts = poptorch.Options()
infer_opts.replicationFactor(REP_FACT)
infer_opts.deviceIterations(DEVICE_ITR)

infer_model = poptorch.inferenceModel(
    test_model,
    options=infer_opts
)

Loading the train model's weights from ./models/best_resnet18_camelyon17_wilds.pt


## Loading test dataset

In [31]:
import torch
import poptorch
import matplotlib.pyplot as plt

# Validation/test dataset
test_data = Camelyon17Dataset(
    test_hf,
    transform=my_transform,
    return_metadata=False
)

test_loader = poptorch.DataLoader(
    infer_opts,
    test_data,
    batch_size=10,
    shuffle=False,
    drop_last=True,
    num_workers=4
)

#  Detach training model from IPU
try:
    poptorch_train.detachFromDevice()
except Exception as e:
    print("detach warning:", e)

detach warning: 'poptorch_py_error': Model has not been compiled or has been destroyed.


## Evaluation

In [32]:
from sklearn.metrics import precision_score, \
recall_score, f1_score, accuracy_score, roc_auc_score, \
confusion_matrix, classification_report
import time

start_time = time.time()

correct = 0
total = 0

all_preds = []
all_labels = []
all_probs = []

#no_images = len(test_data)

with torch.no_grad():
    for images, labels in test_loader:
        logits = infer_model(images)
        
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

# Calculate accuracy
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')

roc_auc = roc_auc_score(all_labels, all_probs)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

print(f"Inference Results of {MODEL_TYPE} with No. Train data={NO_TRAIN_DATA} & No. test data={NO_TEST_DATA} ")
print("------------------------------------------------------------------------------------------")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"AUC: {roc_auc:.4f}")
print("------------------------------------------------------------------------------------------")
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print("------------------------------------------------------------------------------------------")
e_time = time.time()-start_time
n_eval = len(all_labels)
print(f"Elapsed time = {e_time:.3f} secs for {n_eval} images")
print(f"Inference speed = {n_eval/e_time:.1f} images per sec")
print(f"Inference time per image = {e_time/n_eval*1000:.4f} msec per image")

Graph compilation: 100%|██████████| 100/100 [01:20<00:00]


Inference Results of ResNet18 with No. Train data=20000 & No. test data=10000 
------------------------------------------------------------------------------------------
Accuracy: 0.7153
Precision: 0.8194
Recall: 0.7153
F1-Score: 0.6914
AUC: 0.8870
------------------------------------------------------------------------------------------
TN: 4941, FP: 0, FN: 2847, TP: 2212
Sensitivity: 0.4372
Specificity: 1.0000
------------------------------------------------------------------------------------------
Elapsed time = 90.312 secs for 10000 images
Inference speed = 110.7 images per sec
Inference time per image = 9.0312 msec per image


In [ ]:
def display_inference(test_hf, seed=42):
    disp_opts = poptorch.Options()
    disp_opts.replicationFactor(REP_FACT)
    disp_opts.deviceIterations(DEVICE_ITR)
    
    if MODEL_TYPE == "ResNet18":
        display_model = ResNet18Camelyon(num_classes=2, pretrained=False)
        SAVE_MODEL_FILE = "./models/best_resnet18_camelyon17_wilds.pt"
    elif MODEL_TYPE == "ResNet34":
        display_model = ResNet34Camelyon(num_classes=2, pretrained=False)
        SAVE_MODEL_FILE = "./models/best_resnet34_camelyon17_wilds.pt"
    elif MODEL_TYPE == "ResNet50":
        display_model = ResNet50Camelyon(num_classes=2, pretrained=False)
        SAVE_MODEL_FILE = "./models/best_resnet50_camelyon17_wilds.pt"
 
    display_model.load_state_dict(torch.load(SAVE_MODEL_FILE))
    display_model.eval()

    display_infer_model = poptorch.inferenceModel(
        display_model,
        options=disp_opts
    )
    
    # small random inspection set
    random_test_hf = make_random_subset(
        test_hf,
        n_samples=10,
        seed=seed
    )

    # wrap it
    random_test_data = Camelyon17Dataset(
        random_test_hf,
        transform=my_transform,
        return_metadata=False
    )
    
    
    # make it as a loader
    random_loader = poptorch.DataLoader(
        disp_opts,
        random_test_data,
        batch_size=10,
        shuffle=False,
        num_workers=4
    )
    
    # Get one batch
    images, labels = next(iter(random_loader))

    with torch.no_grad():
        logits = display_model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

    # unnormalize ImageNet normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(1, 5, figsize=(15,4))

    for i in range(5):

        img = images[i].cpu()

        img = img * std + mean
        img = img.clamp(0, 1)

        axes[i].imshow(img.permute(1,2,0))
        axes[i].set_title(
            f"True:{labels[i].item()}\n"
            f"Pred.:{preds[i].item()}\n"
            f"Confidence: {probs[i,preds[i]].item():.6f}"
        )
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
display_inference(test_hf, seed=43)
display_inference(test_hf, seed=44)
display_inference(test_hf, seed=45)
display_inference(test_hf, seed=46)

In [ ]:
mistakes = []

for images, labels in test_loader:

    logits = infer_model(images)

    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

    for i in range(len(labels)):
        if preds[i] != labels[i]:
            mistakes.append(
                (
                    images[i].cpu(),
                    labels[i].item(),
                    preds[i].item(),
                    probs[i,1].item()
                )
            )
            
print("Number of mistakes:", len(mistakes))

In [ ]:
import random
import matplotlib.pyplot as plt
import torch

def display_mistakes(mistakes, n_show=5, seed=42):
    random.seed(seed)

    selected = random.sample(
        mistakes,
        k=min(n_show, len(mistakes))
    )

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(1, len(selected), figsize=(3 * len(selected), 4))

    if len(selected) == 1:
        axes = [axes]

    for ax, item in zip(axes, selected):
        img, true_label, pred_label, p_tumor = item

        img = img.cpu()
        img = img * std + mean
        img = img.clamp(0, 1)

        confidence = p_tumor if pred_label == 1 else 1 - p_tumor

        ax.imshow(img.permute(1, 2, 0))
        ax.set_title(
            f"True:{true_label}\n"
            f"Pred:{pred_label}\n"
            f"Tumor:{p_tumor:.6f}\n"
            f"Conf:{confidence:.6f}"
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
display_mistakes(mistakes, n_show=5, seed=123)
display_mistakes(mistakes, n_show=5, seed=124)
display_mistakes(mistakes, n_show=5, seed=125)